# Introducción a ChromaDB

## ¿Por qué una base de datos vectorial?

| | BD Relacional (SQL) | BD Vectorial |
|---|---|---|
| **Tipo de dato** | Filas/columnas estructuradas | Vectores de alta dimensión |
| **Tipo de consulta** | Coincidencia exacta, rango, unión | Vecino más cercano (similitud) |
| **Caso de uso** | Facturas, usuarios, pedidos | Embeddings, búsqueda semántica, reconocimiento facial |
| **Búsqueda** | `WHERE id = 42` | 'Encontrar los 5 embeddings más similares a esta consulta' |

Una base de datos vectorial almacena embeddings y los recupera **por similitud** — no por valor exacto. Los embeddings de CLIP que generamos en el notebook anterior son vectores de 512 dimensiones. Una vez almacenados en una base de datos vectorial, podemos preguntar: *'¿Qué cara inscrita es más similar a esta nueva foto?'* — y la base de datos responde en milisegundos, incluso con millones de embeddings almacenados.

En este notebook usamos **ChromaDB**, una base de datos vectorial de código abierto escrita en Python que no requiere servidor externo y se instala como un paquete pip.

## Conceptos Fundamentales

| Término | Significado |
|------|---------| 
| **Client** | Punto de entrada — gestiona la conexión a la base de datos y las colecciones |
| **Collection** | Grupo con nombre de embeddings (como una tabla en SQL) |
| **ID** | Identificador de cadena único para cada fila |
| **Embedding** | El vector almacenado en la colección |
| **Document** | Texto original opcional asociado al embedding (útil para tareas de texto) |
| **Metadata** | Pares clave-valor arbitrarios almacenados junto a cada embedding (p. ej. timestamp, etiqueta) |

Una sola fila en una colección = (ID, embedding, [document], [metadata]).

## Instalación y Configuración

In [ ]:
# pip install chromadb  (or: uv add chromadb)import chromadbprint(f'ChromaDB version: {chromadb.__version__}')

ChromaDB version: 1.5.5


`EphemeralClient()` crea una base de datos en memoria que solo existe mientras el proceso Python actual está activo. Por eso no aparecen archivos de base de datos en disco.

Esto es útil para enseñanza y experimentos rápidos porque mantiene la configuración simple.

Usa `PersistentClient(path='...')` cuando quieras que la colección sobreviva después de que el notebook deje de ejecutarse. En los notebooks de aplicación posteriores, la persistencia importa porque las caras inscritas o los embeddings almacenados deben seguir existiendo en una nueva sesión.

In [ ]:
# EphemeralClient: solo en memoria — los datos se pierden al terminar el proceso.
# Usa PersistentClient(path='./my_db') para guardar en disco.

## NOTA: EphemeralClient comparte el mismo almacén en memoria dentro de una
# sesión del kernel, por lo que create_collection() lanza InternalError si se
# vuelve a ejecutar la celda. get_or_create_collection() es idempotente: crea
# en la primera ejecución, devuelve la colección existente en las siguientes.
client = chromadb.EphemeralClient()

# Crea o recupera una colección. distance_function controla cómo se mide la similitud.
# 'cosine' es la elección correcta para embeddings normalizados (p. ej. CLIP).
collection = client.get_or_create_collection(
    name='demo_embeddings',
    metadata={'hnsw:space': 'cosine'}  # métrica de distancia: coseno
)

print(f'Collection: {collection.name}')
print(f'Current count: {collection.count()}')

## Añadir Datos

Usa `collection.add()` para insertar una o más filas.
Cada fila necesita como mínimo un **ID** y un **embedding**.

In [ ]:
import numpy as np

# Usaremos embeddings sintéticos de 4 dimensiones para mayor claridad.
# En un sistema real serían salidas de CLIP de 512 dimensiones.
rng = np.random.default_rng(42)  # semilla fija para reproducibilidad

# Simular embeddings faciales para 3 estudiantes
alice_emb  = rng.standard_normal(4).tolist()
bob_emb    = rng.standard_normal(4).tolist()
charlie_emb = rng.standard_normal(4).tolist()

collection.add(
    ids        = ['alice_001', 'bob_001', 'charlie_001'],
    embeddings = [alice_emb, bob_emb, charlie_emb],
    metadatas  = [
        {'name': 'Alice',   'session': 'enroll_2024',  'frame': 0},
        {'name': 'Bob',     'session': 'enroll_2024',  'frame': 0},
        {'name': 'Charlie', 'session': 'enroll_2024',  'frame': 0},
    ])

print(f'Collection now has {collection.count()} embeddings')

In [ ]:
# Añadir más embeddings para Alice (p. ej. de múltiples capturas en la inscripción)
# El pequeño ruido simula a la misma persona fotografiada dos veces
alice_emb2 = (np.array(alice_emb) + rng.standard_normal(4) * 0.05).tolist()
alice_emb3 = (np.array(alice_emb) + rng.standard_normal(4) * 0.05).tolist()

collection.add(
    ids        = ['alice_002', 'alice_003'],
    embeddings = [alice_emb2, alice_emb3],
    metadatas  = [
        {'name': 'Alice', 'session': 'enroll_2024', 'frame': 1},
        {'name': 'Alice', 'session': 'enroll_2024', 'frame': 2},
    ])

print(f'Total embeddings: {collection.count()}')

## Consulta por Similitud

`collection.query()` encuentra los **K vecinos más cercanos** de un embedding de consulta.
ChromaDB devuelve **distancias** (no similitudes) — menor distancia significa más similitud. Para la métrica coseno:

$$d_{\text{cosine}}(a, b) = 1 - \frac{a \cdot b}{\|a\| \cdot \|b\|}$$

- distancia = 0 → los vectores apuntan en la misma dirección → coincidencia perfecta
- distancia = 1 → los vectores son ortogonales → sin similitud
- distancia = 2 → los vectores apuntan en direcciones opuestas (raro en embeddings reales)

Para convertir de nuevo a similitud: $\text{similitud} = 1 - d$.

In [ ]:
# Simular a Alice presentándose en verificación: su embedding con pequeña perturbación
alice_query = (np.array(alice_emb) + rng.standard_normal(4) * 0.1).tolist()

results = collection.query(
    query_embeddings = [alice_query],   # lista de vectores de consulta
    n_results = 3,                       # devolver los 3 más similares
    include = ['metadatas', 'distances'],
)

print('Resultados top-3 para consulta de Alice:')
for rank, (doc_id, dist, meta) in enumerate(zip(
        results['ids'][0],
        results['distances'][0],
        results['metadatas'][0])):
    similarity = 1 - dist  # convertir distancia coseno a similitud
    print(f'  [{rank+1}] id={doc_id:12s}  cosine_similarity={similarity:.4f}  name={meta["name"]}')

## Filtrado por Metadatos

El argumento `where` filtra los resultados por metadatos **antes** de la búsqueda por similitud.
Esto permite buscar solo dentro de un grupo específico (p. ej. la sesión de un solo estudiante).

In [ ]:
# Buscar solo dentro de las entradas de inscripción de Alice
results_filtered = collection.query(
    query_embeddings = [alice_query],
    n_results = 2,
    where = {'name': 'Alice'},          # solo entradas donde el metadata 'name' == 'Alice'
    include = ['metadatas', 'distances'],
)

print('Filtrado solo para Alice:')
for doc_id, dist, meta in zip(
        results_filtered['ids'][0],
        results_filtered['distances'][0],
        results_filtered['metadatas'][0]):
    print(f'  id={doc_id:12s}  similarity={1-dist:.4f}  frame={meta["frame"]}')

## Obtener y Eliminar Entradas

In [ ]:
# Recuperar una entrada específica por ID
result = collection.get(
    ids=['alice_001'],
    include=['embeddings', 'metadatas'])
print('alice_001 metadata:', result['metadatas'][0])
print('alice_001 embedding primeros 4 valores:', result['embeddings'][0][:4])

# Eliminar una entrada
collection.delete(ids=['charlie_001'])
print(f'Después de eliminar charlie_001: {collection.count()} entradas restantes')

## Cómo Funciona la Búsqueda por Similitud Internamente

La búsqueda por similitud por fuerza bruta escala como **O(N × D)**: para N=1M embeddings de dimensión D=512 eso son 512 millones de multiplicaciones por consulta — demasiado lento a gran escala.

ChromaDB usa **HNSW** (Hierarchical Navigable Small World), un algoritmo de grafo para Vecino Más Cercano Aproximado (ANN).

**Intuición:**
1. Los embeddings se organizan en un grafo de múltiples capas donde cada nodo se conecta con sus vecinos más cercanos.
2. Una consulta comienza en la capa superior (dispersa) y recorre el grafo avidamente moviéndose siempre hacia el vecino más similar a la consulta.
3. Esto reduce la búsqueda a un pequeño conjunto de candidatos, que se re-ordena exactamente.

El resultado es **aproximado** (ocasionalmente el verdadero vecino más cercano puede perderse) pero **rápido** — el tiempo de consulta típico es O(log N) en lugar de O(N).

Para conjuntos de datos de enseñanza con unas pocas docenas de embeddings, la diferencia es insignificante en la práctica. Para sistemas de producción con millones de caras o documentos, HNSW o algoritmos similares (FAISS, ScaNN) son esenciales.

El parámetro `hnsw:space` en los metadatos de la colección debe coincidir con la geometría con la que se entrenaron los embeddings. Los vectores de CLIP están normalizados en L2 (norma unitaria), lo que hace equivalentes la distancia coseno y el producto interno — pero la colección aún debe saber cuál usar.

## Métricas de Distancia

Al crear una colección, elige la métrica que coincida con cómo se entrenaron los embeddings:

| Métrica | `hnsw:space` | Mejor para |
|--------|-------------|----------|
| Distancia coseno | `'cosine'` | Embeddings normalizados (CLIP, Sentence-BERT, la mayoría de modelos modernos) |
| L2 / Euclidiana | `'l2'` | Vectores de características crudas donde la magnitud importa |
| Producto interno | `'ip'` | Búsqueda de máximo producto interno; más rápido cuando los vectores están pre-normalizados |

> Para embeddings de CLIP, usa **cosine** — el modelo fue entrenado con la similitud coseno como objetivo.

In [ ]:
# Demostrar las tres opciones de métricas (solo colecciones, sin datos añadidos)
client2 = chromadb.EphemeralClient()
col_cosine = client2.create_collection('test_cosine', metadata={'hnsw:space': 'cosine'})
col_l2     = client2.create_collection('test_l2',     metadata={'hnsw:space': 'l2'})
col_ip     = client2.create_collection('test_ip',     metadata={'hnsw:space': 'ip'})
print('Collections creadas:', [c.name for c in client2.list_collections()])

## Resumen

| Operación | Código |
|-----------|------|
| Crear cliente | `chromadb.EphemeralClient()` / `PersistentClient(path)` |
| Crear colección | `client.create_collection(name, metadata={'hnsw:space': 'cosine'})` |
| Añadir embeddings | `collection.add(ids, embeddings, metadatas)` |
| Consultar por similitud | `collection.query(query_embeddings, n_results, where)` |
| Obtener por ID | `collection.get(ids)` |
| Eliminar por ID | `collection.delete(ids)` |
| Contar entradas | `collection.count()` |

La idea principal es simple: los embeddings solo son útiles a escala cuando podemos recuperar sus vecinos más cercanos de forma eficiente.

Esto importa para el reconocimiento facial, la recuperación de imágenes, y también para sistemas de NLP que almacenan embeddings de frases o documentos. El siguiente notebook combina detección, embeddings y búsqueda vectorial en un pipeline completo de reconocimiento facial.